# Predição de Reinternação Hospitalar de Pacientes Diabéticos em menos de 30 dias

**Disciplina:** Ciência de Dados e Big Data

| Item | Informação |
|---|---|
| **Integrantes** | Bernardo Dauber — *(nome do(a) integrante 2)* |
| **Dataset** | [Diabetes 130-US Hospitals for Years 1999-2008 (UCI)](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008) |
| **Apresentação (slides)** | *(colar link — garantir acesso ao professor)* |
| **Vídeo (máx. 10 min)** | *(colar link — garantir acesso ao professor)* |

### 🎬 Guia de apresentação (vídeo gravado neste notebook)

O notebook foi executado do início ao fim antes da gravação. Em cada etapa, mostramos a saída e lemos o bloco **🎤 O que encontramos** logo abaixo dela. A apresentação resumida em slides está linkada na célula acima.

| Parte | Seções | Quem apresenta |
|---|---|---|
| Problema, dados e exploração | 1 a 6 | Integrante 1 |
| Limpeza e faltantes | 7 e 8 | Integrante 1 |
| Atributos, tipos, X e y | 9 a 11 | Integrante 2 |
| Treino/teste, padronização, modelos e resultados | 12 a 15 | Integrante 2 |
| Análise crítica e fechamento | 16 | Os dois |

## 1. Contexto e problema

**Contexto.** A reinternação precoce de pacientes diabéticos é um indicador de qualidade do cuidado e gera custos elevados para sistemas de saúde. Identificar pacientes com maior risco de voltar ao hospital em menos de 30 dias permite priorizar acompanhamento pós-alta.

**Pergunta.** *É possível prever, a partir de dados da internação (perfil do paciente, procedimentos, medicamentos, diagnósticos), se um paciente diabético será reinternado em menos de 30 dias?*

**Tipo de problema.** Classificação supervisionada binária (`1` = reinternado em < 30 dias, `0` = caso contrário).

**Dataset.** ~100 mil internações em 130 hospitais dos EUA (1999–2008), ~50 variáveis (numéricas e categóricas), com valores faltantes e inconsistentes.

> TODO (dupla): reescrever esta seção com as palavras de vocês e acrescentar a motivação pessoal/social da escolha.

## 2. Importação das bibliotecas

In [ ]:
import io
import os
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, roc_auc_score, RocCurveDisplay)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42

## 3. Aquisição e importação dos dados *(etapas 1 e 2)*

- **Fonte:** UCI Machine Learning Repository — https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008
- O arquivo principal é `diabetic_data.csv`; `IDs_mapping.csv` traz o significado dos códigos de admissão/alta.

A célula abaixo baixa e extrai o `.zip` automaticamente. Se o download falhar, baixe manualmente pelo link acima e envie `diabetic_data.csv` para o Colab.

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"
DATA_DIR = "data"
CSV_PATH = os.path.join(DATA_DIR, "diabetic_data.csv")

if not os.path.exists(CSV_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    try:
        req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp:
            with zipfile.ZipFile(io.BytesIO(resp.read())) as z:
                z.extractall(DATA_DIR)
        print("Download e extração concluídos.")
    except Exception as e:
        print("Falha no download automático:", e)
        print("Baixe manualmente e coloque diabetic_data.csv em ./data/")

df_raw = pd.read_csv(CSV_PATH)
df = df_raw.copy()   # trabalhamos numa cópia; df_raw fica intacto para comparações
print("Importado com sucesso:", df.shape)

## 4. Análise exploratória inicial *(etapa 3)*

In [ ]:
print("Dimensões (linhas, colunas):", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Estatísticas das colunas categóricas
df.describe(include="object").T

In [ ]:
# Valores faltantes: neste dataset eles aparecem como '?' (não como NaN)
missing_q = (df == "?").sum()
missing_q = missing_q[missing_q > 0].sort_values(ascending=False).to_frame("qtd_?")
missing_q["%"] = (missing_q["qtd_?"] / len(df) * 100).round(2)
print("Duplicatas de linhas inteiras:", df.duplicated().sum())
print("Pacientes únicos:", df["patient_nbr"].nunique(), "de", len(df), "internações")
missing_q

#### 🎤 O que encontramos — dados brutos

> - O dataset tem **[__] linhas × [__] colunas**, com tipos numéricos e categóricos.
> - Os faltantes aparecem como `?`; `weight` tem **[__]%**, `medical_specialty` **[__]%** e `payer_code` **[__]%**.
> - Há **[__] pacientes únicos** em [__] internações: o mesmo paciente aparece várias vezes.

## 5. Entendimento dos dados e das variáveis *(etapa 4)*

Dicionário resumido (fonte: UCI / artigo original *Strack et al., 2014*):

| Grupo | Variáveis | Descrição |
|---|---|---|
| Identificadores | `encounter_id`, `patient_nbr` | ID da internação e do paciente (**não são preditores**) |
| Demográficas | `race`, `gender`, `age`, `weight` | Perfil do paciente (`age` em faixas de 10 anos) |
| Admissão/alta | `admission_type_id`, `admission_source_id`, `discharge_disposition_id` | Códigos categóricos (ver `IDs_mapping.csv`) |
| Cuidado | `time_in_hospital`, `num_lab_procedures`, `num_procedures`, `num_medications` | Contagens/duração da internação |
| Histórico | `number_outpatient`, `number_emergency`, `number_inpatient` | Visitas no ano anterior |
| Diagnóstico | `diag_1`, `diag_2`, `diag_3`, `number_diagnoses` | Códigos CID-9 e quantidade de diagnósticos |
| Exames | `max_glu_serum`, `A1Cresult` | Glicose sérica e HbA1c |
| Medicamentos | 23 colunas (`metformin`, `insulin`, ...) | `No`, `Steady`, `Up`, `Down` |
| Gerais | `change`, `diabetesMed`, `payer_code`, `medical_specialty` | Mudança de medicação, uso de antidiabéticos, convênio, especialidade |
| **Alvo** | `readmitted` | `NO`, `>30`, `<30` dias |

> TODO (dupla): descrever em texto corrido o que cada grupo significa clinicamente e quais variáveis vocês esperam que mais influenciem o alvo (hipóteses — voltaremos a elas na conclusão).

### 5.1 Definição da variável-alvo

Transformamos `readmitted` em **binária**: `1` se `<30` (reinternação precoce), `0` caso contrário (`>30` ou `NO`). Isso reflete a métrica usada por hospitais (readmissão em 30 dias).

In [ ]:
df["target"] = (df["readmitted"] == "<30").astype(int)
print(df["readmitted"].value_counts())
print("\nProporção do alvo binário:")
print(df["target"].value_counts(normalize=True).round(3))

## 6. Visualização dos dados *(etapa 5)*

Gráficos exploratórios sobre o alvo, distribuições numéricas, categóricas e relações entre variáveis.

In [ ]:
# 6.1 Distribuição do alvo (desbalanceamento)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df, x="readmitted", order=["NO", ">30", "<30"], ax=ax[0])
ax[0].set_title("Reinternação (original, 3 classes)")
sns.countplot(data=df, x="target", ax=ax[1])
ax[1].set_title("Alvo binário (1 = reinternado < 30 dias)")
plt.tight_layout(); plt.show()

In [ ]:
# 6.2 Histogramas das variáveis numéricas
num_cols = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
            "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
df[num_cols].hist(bins=30, figsize=(14, 8))
plt.suptitle("Distribuição das variáveis numéricas", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# 6.3 Boxplots (outliers) das variáveis numéricas
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), num_cols):
    sns.boxplot(y=df[c], ax=ax)
    ax.set_title(c)
plt.tight_layout(); plt.show()

In [ ]:
# 6.4 Taxa de reinternação < 30 dias por faixa etária e por nº de internações prévias
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
(df.groupby("age")["target"].mean() * 100).plot(kind="bar", ax=ax[0])
ax[0].set_ylabel("% reinternados < 30d"); ax[0].set_title("Por faixa etária")
(df.groupby(df["number_inpatient"].clip(upper=8))["target"].mean() * 100).plot(kind="bar", ax=ax[1])
ax[1].set_ylabel("% reinternados < 30d"); ax[1].set_title("Por nº de internações prévias (>=8 agrupado)")
plt.tight_layout(); plt.show()

In [ ]:
# 6.5 Boxplot de variável numérica x alvo + dispersão
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df, x="target", y="time_in_hospital", ax=ax[0])
ax[0].set_title("Tempo de internação x reinternação < 30d")
sns.scatterplot(data=df.sample(5000, random_state=RANDOM_STATE),
                x="num_medications", y="num_lab_procedures", hue="target", alpha=0.5, ax=ax[1])
ax[1].set_title("Medicamentos x exames laboratoriais (amostra de 5 mil)")
plt.tight_layout(); plt.show()

In [ ]:
# 6.6 Correlação entre variáveis numéricas
plt.figure(figsize=(9, 7))
sns.heatmap(df[num_cols + ["target"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlação"); plt.show()

#### 🎤 O que encontramos — gráficos

> - Só **[__]%** das internações terminam em reinternação em menos de 30 dias: o alvo é desbalanceado.
> - As variáveis `number_*` são muito assimétricas, com outliers: [descrever o que o boxplot mostrou].
> - [Padrão observado entre internações prévias, idade ou tempo de internação e o alvo.]
> - As correlações entre as numéricas são [fracas / moderadas]: [maior correlação observada = __].

## 7. Tratamento de dados inválidos ou inconsistentes *(etapa 6)*

Problemas identificados e decisões:

1. **`?`** representa ausência → converter para `NaN` (tratado na seção 8).
2. **`gender = 'Unknown/Invalid'`** (poucos registros) → remover.
3. **Óbitos/hospice** (`discharge_disposition_id` ∈ {11, 13, 14, 19, 20, 21}): esses pacientes **não podem ser reinternados** → remover para não distorcer o modelo.
4. **Múltiplas internações do mesmo paciente** violam a independência das amostras e podem causar *vazamento* entre treino e teste → manter apenas a **primeira internação por paciente**.
5. **Colunas constantes** (ex.: `examide`, `citoglipton` têm apenas 'No') → serão descartadas na seleção de atributos.

In [ ]:
n0 = len(df)

# 1) '?' -> NaN
df = df.replace("?", np.nan)

# 2) gênero inválido
df = df[df["gender"] != "Unknown/Invalid"]

# 3) óbito / hospice
obito_hospice = [11, 13, 14, 19, 20, 21]
df = df[~df["discharge_disposition_id"].isin(obito_hospice)]

# 4) uma internação por paciente (a primeira, por encounter_id)
df = df.sort_values("encounter_id").drop_duplicates(subset="patient_nbr", keep="first")

print(f"Linhas antes: {n0:,} | depois: {len(df):,} (removidas {n0 - len(df):,})")
print("Taxa do alvo após limpeza:", df["target"].mean().round(4))

#### 🎤 O que encontramos — dados inválidos

> - Passamos de **[__] para [__] linhas** depois de remover gênero inválido, óbito/hospice e internações repetidas.
> - A taxa do alvo foi de **[__]%** para **[__]%** depois da limpeza.

## 8. Tratamento de dados faltantes *(etapa 7)*

Estratégia por coluna, baseada no percentual de faltantes:

| Coluna | Estratégia | Justificativa |
|---|---|---|
| `weight` | **Remover coluna** | ~97% faltante: imputar seria inventar dado |
| `payer_code`, `medical_specialty` | Categoria `"Desconhecido"` | ~40–50% faltante; a ausência pode ser informativa |
| `race`, `diag_1/2/3` | Categoria `"Desconhecido"` | Poucos faltantes; preserva as linhas |
| `max_glu_serum`, `A1Cresult` | Já são `None`/`Norm`/`>X` (exame não realizado é categoria válida) | Não são `NaN` |

In [ ]:
miss = (df.isna().mean() * 100).round(2)
miss[miss > 0].sort_values(ascending=False)

In [ ]:
df = df.drop(columns=["weight"])

cols_desconhecido = ["payer_code", "medical_specialty", "race", "diag_1", "diag_2", "diag_3"]
df[cols_desconhecido] = df[cols_desconhecido].fillna("Desconhecido")

print("Faltantes restantes:", int(df.isna().sum().sum()))

#### 🎤 O que encontramos — faltantes

> - `weight` foi removida (faltante em cerca de 97%); as demais colunas viraram a categoria “Desconhecido”.
> - Faltantes restantes: **[__]** (esperado: 0).

## 9. Engenharia e seleção de atributos *(etapa 8)*

- **Remover identificadores:** `encounter_id`, `patient_nbr` (não generalizam).
- **Remover `readmitted`:** é a origem do alvo (*vazamento direto*).
- **Remover colunas constantes/quase constantes** (ex.: `examide`, `citoglipton`).
- **Agrupar diagnósticos CID-9** (`diag_1/2/3` têm centenas de códigos) em categorias clínicas.
- **`age`** é ordinal → converter faixa para o ponto médio (ex.: `[70-80)` → 75).
- **`medical_specialty`** e **`payer_code`** têm alta cardinalidade → manter apenas as categorias mais frequentes e agrupar o resto em `Outros`.

In [ ]:
# 9.1 Descartar identificadores, vazamento e colunas constantes
df = df.drop(columns=["encounter_id", "patient_nbr", "readmitted"])

constantes = [c for c in df.columns if df[c].nunique() <= 1]
print("Colunas constantes removidas:", constantes)
df = df.drop(columns=constantes)

# Colunas de medicamentos com variância quase nula (>99,5% 'No')
meds = ["metformin","repaglinide","nateglinide","chlorpropamide","glimepiride","acetohexamide",
        "glipizide","glyburide","tolbutamide","pioglitazone","rosiglitazone","acarbose","miglitol",
        "troglitazone","tolazamide","insulin","glyburide-metformin","glipizide-metformin",
        "glimepiride-pioglitazone","metformin-rosiglitazone","metformin-pioglitazone"]
meds = [m for m in meds if m in df.columns]
quase_const = [m for m in meds if (df[m] == "No").mean() > 0.995]
print("Medicamentos quase constantes removidos:", quase_const)
df = df.drop(columns=quase_const)

In [ ]:
# 9.2 Agrupar códigos CID-9 em categorias clínicas
def agrupar_diag(code):
    if code == "Desconhecido":
        return "Desconhecido"
    if str(code).startswith(("V", "E")):
        return "Outros"
    try:
        v = float(code)
    except ValueError:
        return "Outros"
    if 390 <= v <= 459 or v == 785: return "Circulatório"
    if 460 <= v <= 519 or v == 786: return "Respiratório"
    if 520 <= v <= 579 or v == 787: return "Digestivo"
    if int(v) == 250:               return "Diabetes"
    if 800 <= v <= 999:             return "Lesões"
    if 710 <= v <= 739:             return "Musculoesquelético"
    if 580 <= v <= 629 or v == 788: return "Geniturinário"
    if 140 <= v <= 239:             return "Neoplasias"
    return "Outros"

for c in ["diag_1", "diag_2", "diag_3"]:
    df[c] = df[c].apply(agrupar_diag)

# 9.3 Idade: faixa -> ponto médio (variável ordinal tratada como numérica)
df["age"] = df["age"].str.extract(r"\[(\d+)-(\d+)\)").astype(int).mean(axis=1)

# 9.4 Alta cardinalidade: manter as 10 categorias mais frequentes
for c in ["medical_specialty", "payer_code"]:
    top = df[c].value_counts().nlargest(10).index
    df[c] = df[c].where(df[c].isin(top), "Outros")

# 9.5 Códigos numéricos que na verdade são categorias
for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    df[c] = df[c].astype(str)

print("Shape após seleção/engenharia:", df.shape)
df.head()

## 10. Classificação das variáveis *(etapa 9)*

| Tipo | Variáveis |
|---|---|
| **Numéricas discretas** (contagens) | `time_in_hospital`, `num_lab_procedures`, `num_procedures`, `num_medications`, `number_outpatient`, `number_emergency`, `number_inpatient`, `number_diagnoses` |
| **Numérica ordinal tratada como contínua** | `age` (ponto médio da faixa) |
| **Categóricas nominais** | `race`, `gender`, `admission_type_id`, `discharge_disposition_id`, `admission_source_id`, `payer_code`, `medical_specialty`, `diag_1/2/3`, `change`, `diabetesMed`, medicamentos restantes |
| **Categóricas ordinais** | `max_glu_serum`, `A1Cresult` (`None` < `Norm` < `>7`/`>200` < `>8`/`>300`) — por simplicidade tratadas como nominais via One-Hot |
| **Alvo (binária)** | `target` |

> Nenhuma variável é contínua no sentido estrito (todas as numéricas são contagens/durações inteiras, exceto `age` que foi derivada).

In [ ]:
TARGET = "target"
num_features = ["age"] + [c for c in df.columns if c.startswith(("time_", "num_", "number_"))]
cat_features = [c for c in df.columns if c not in num_features + [TARGET]]

print("Numéricas  (%d):" % len(num_features), num_features)
print("Categóricas (%d):" % len(cat_features), cat_features)

#### 🎤 O que encontramos — atributos

> - Ficamos com **[__] numéricas** e **[__] categóricas**, depois de remover IDs, `readmitted` e colunas constantes.
> - Colunas de medicamentos removidas por serem quase constantes: [__].

## 11. Separação entre preditores (X) e alvo (y) *(etapa 10)*

In [ ]:
X = df[num_features + cat_features]
y = df[TARGET]
print("X:", X.shape, "| y:", y.shape)

## 12. Divisão em treino e teste *(etapa 13)*

Divisão **antes** de qualquer escalonamento/codificação ajustada, para evitar *data leakage*. Usamos `stratify=y` para manter a proporção do alvo (desbalanceado) em ambos os conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("Proporção do alvo — treino: %.3f | teste: %.3f" % (y_train.mean(), y_test.mean()))

## 13. Transformações e padronização *(etapas 11 e 12)*

Usamos um `ColumnTransformer` dentro de um `Pipeline`:

- **Numéricas:** `StandardScaler` (média 0, desvio 1) — necessário para a Regressão Logística.
- **Categóricas:** `OneHotEncoder(handle_unknown="ignore")` — transforma categorias em colunas binárias.

O pré-processador é **ajustado apenas no treino** (`fit`) e aplicado ao teste (`transform`) automaticamente pelo pipeline.

In [ ]:
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

# Visualizar o efeito da transformação (apenas para inspeção)
Xt = preprocess.fit_transform(X_train)
print("Shape após transformação:", Xt.shape)

#### 🎤 O que encontramos — transformações

> - Depois do `OneHotEncoder` o conjunto de treino tem **[__] colunas**.
> - O pré-processador é ajustado só no treino, o que evita vazamento para o teste.

## 14. Treinamento dos modelos *(etapa 14)*

Dois modelos para permitir comparação crítica:

1. **Regressão Logística** — *baseline* interpretável.
2. **Random Forest** — modelo não linear, captura interações e é robusto a outliers.

Como o alvo é desbalanceado (~9–11% de positivos), usamos `class_weight="balanced"` para penalizar mais os erros na classe minoritária.

In [ ]:
modelos = {
    "Regressão Logística": LogisticRegression(max_iter=1000, class_weight="balanced",
                                              random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=20,
                                            class_weight="balanced", n_jobs=-1,
                                            random_state=RANDOM_STATE),
}

pipes = {}
for nome, clf in modelos.items():
    pipes[nome] = Pipeline([("prep", preprocess), ("clf", clf)])
    pipes[nome].fit(X_train, y_train)
    print(f"{nome}: treinado.")

## 15. Avaliação dos modelos *(etapa 15)*

**Atenção à métrica:** com ~10% de positivos, um modelo que sempre prevê "não reinterna" teria ~90% de acurácia e seria inútil. Por isso olhamos **recall** (quantos reinternados detectamos), **precisão**, **F1** e **ROC-AUC**.

In [ ]:
# Baseline trivial para referência (sempre prevê a classe majoritária)
print("Acurácia do 'modelo burro' (sempre 0): %.3f" % (1 - y_test.mean()))

resultados = []
for nome, pipe in pipes.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    resultados.append({
        "Modelo": nome,
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })
pd.DataFrame(resultados).set_index("Modelo").round(3)

In [ ]:
# Matrizes de confusão
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (nome, pipe) in zip(axes, pipes.items()):
    ConfusionMatrixDisplay.from_estimator(pipe, X_test, y_test, ax=ax, cmap="Blues",
                                          display_labels=["Não reinterna", "Reinterna <30d"])
    ax.set_title(nome)
plt.tight_layout(); plt.show()

In [ ]:
# Relatório de classificação e curva ROC
for nome, pipe in pipes.items():
    print("=" * 60); print(nome); print("=" * 60)
    print(classification_report(y_test, pipe.predict(X_test),
                                target_names=["Não reinterna", "Reinterna <30d"]))

fig, ax = plt.subplots(figsize=(6, 5))
for nome, pipe in pipes.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=nome, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Aleatório")
ax.set_title("Curva ROC"); ax.legend(); plt.show()

In [ ]:
# Importância das variáveis (Random Forest) — interpretabilidade
rf = pipes["Random Forest"]
nomes = rf.named_steps["prep"].get_feature_names_out()
imp = pd.Series(rf.named_steps["clf"].feature_importances_, index=nomes).nlargest(15)
imp.sort_values().plot(kind="barh", figsize=(8, 6), title="Top 15 variáveis — Random Forest")
plt.tight_layout(); plt.show()

#### 🎤 O que encontramos — resultados

> - **Regressão Logística:** acurácia [__], precisão [__], recall [__], F1 [__], ROC-AUC [__].
> - **Random Forest:** acurácia [__], precisão [__], recall [__], F1 [__], ROC-AUC [__].
> - O modelo que sempre prevê “não” teria acurácia de **[__]**: por isso olhamos recall e F1.
> - A matriz de confusão do melhor modelo tem **[__] falsos negativos** e **[__] falsos positivos**.
> - Variáveis mais importantes: [__], [__], [__]. Isso [confirma / contradiz] a hipótese da seção 5.

## 16. Conclusões e análise crítica

> TODO (dupla): preencher com base nos resultados reais após executar o notebook.

**Pontos a discutir:**
- **Qual modelo foi melhor e por quê?** Comparar recall/F1/AUC, não só acurácia.
- **O desempenho é suficiente?** Em geral, esse problema tem AUC modesta (~0,60–0,68): os dados de internação sozinhos explicam pouco da reinternação.
- **Trade-off precisão x recall:** em saúde, o custo de um falso negativo (paciente de risco não detectado) costuma ser maior que o de um falso positivo. Discutir o ajuste do limiar de decisão.
- **Variáveis mais importantes:** confirmam ou contradizem as hipóteses da seção 5? (esperado: `number_inpatient`, `number_diagnoses`, `discharge_disposition_id`, `num_medications`).
- **Limitações:** dados de 1999–2008; só hospitais dos EUA; sem dados socioeconômicos ou de adesão ao tratamento; desbalanceamento; uma internação por paciente reduz o volume.
- **Vieses/ética:** uso de `race` e `gender` em modelos clínicos exige cautela; risco de discriminação.
- **Melhorias futuras:** `GridSearchCV`/`RandomizedSearchCV`, validação cruzada, SMOTE, Gradient Boosting (XGBoost/LightGBM), ajuste de limiar, mais engenharia de atributos.

**Referências**
- Strack, B. et al. (2014). *Impact of HbA1c Measurement on Hospital Readmission Rates: Analysis of 70,000 Clinical Database Patient Records.* BioMed Research International.
- UCI Machine Learning Repository — Diabetes 130-US hospitals for years 1999-2008.